In [1]:
# Basic data libraries
import pandas as pd
import numpy as np

# Visualization (optional but helpful)
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load datasets
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

# Check the data
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (8693, 14)
Test shape: (4277, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [2]:
# Basic overview
train_df.info()
train_df.describe()

# Check missing values
train_df.isnull().sum().sort_values(ascending=False)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


CryoSleep       217
ShoppingMall    208
VIP             203
HomePlanet      201
Name            200
Cabin           199
VRDeck          188
Spa             183
FoodCourt       183
Destination     182
RoomService     181
Age             179
PassengerId       0
Transported       0
dtype: int64

In [3]:
spend_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
train_df[spend_cols] = train_df[spend_cols].fillna(0)
test_df[spend_cols]  = test_df[spend_cols].fillna(0)


In [8]:
train_df.isnull().sum().sort_values(ascending=False)


CryoSleep       217
VIP             203
Name            200
Cabin_num       199
Cabin           199
Age             179
PassengerId       0
Destination       0
HomePlanet        0
FoodCourt         0
RoomService       0
Spa               0
ShoppingMall      0
VRDeck            0
Transported       0
Deck              0
Side              0
dtype: int64

In [4]:
train_df[['Deck','Cabin_num','Side']] = train_df['Cabin'].str.split('/', expand=True)
test_df[['Deck','Cabin_num','Side']] = test_df['Cabin'].str.split('/', expand=True)


In [5]:
for col in ['HomePlanet','Destination','Deck','Side']:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))


In [6]:
X = train_df.drop(['Transported','Name','Cabin','PassengerId'], axis=1)
y = train_df['Transported']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier()
model.fit(X_train, y_train)
pred = model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, pred))


Accuracy: 0.7958596894767107


In [7]:
test_pred = model.predict(test_df.drop(['Name','Cabin','PassengerId'], axis=1))

submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Transported': test_pred
})

submission.to_csv("submission.csv", index=False)
submission.head()


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


In [9]:
submission.head()
submission.tail()


,PassengerId,Transported
4272,9266_02,False
4273,9269_01,False
4274,9271_01,True
4275,9273_01,True
4276,9277_01,False
